# RAG with HuggingFace and Milvus - Student Notebook

Here I implemented a complete RAG (Retrieval-Augmented Generation) pipeline using:
- **Dataset**: HuggingFace Documentation (`m-ric/huggingface_doc`)
- **Vector Store**: Milvus
- **Embeddings**: BGE-small-en-v1.5
- **LLM**: Microsoft Phi-3-mini-4k-instruct/"Qwen/Qwen2-1.5B-Instruct"
- **Evaluation**: Opik (AnswerRelevance, Hallucination)



##https://github.com/milvus-io/milvus

## 1. Setup

Installed required dependencies and configure environment.

In [ ]:
# Install dependencies
!pip install -q --upgrade transformers pymilvus sentence-transformers datasets torch accelerate opik tqdm openai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 46.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.6.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.


In [ ]:
print('Attempting to install specific compatible versions of libraries...')
!pip uninstall -y transformers sentence-transformers torch accelerate
!pip install -q --force-reinstall --no-cache-dir torch==2.13.0 accelerate==0.21.0
!pip install -q --force-reinstall --no-cache-dir transformers==4.38.2 sentence-transformers==2.2.2
print('Specific version installation complete. Please re-run the subsequent cells.')

Attempting to install specific compatible versions of libraries...
Found existing installation: transformers 5.17.0
Uninstalling transformers-5.17.0:
  Successfully uninstalled transformers-5.17.0
Found existing installation: sentence-transformers 6.0.1
Uninstalling sentence-transformers-6.0.1:
  Successfully uninstalled sentence-transformers-6.0.1
Found existing installation: torch 2.14.0
Uninstalling torch-2.14.0:
  Successfully uninstalled torch-2.14.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 190.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.2/244.2 kB 291.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 143.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 155.1 MB/s eta 0:00:00
   ━━━━━━━━

In [ ]:
import os
import json
from typing import List, Dict, Tuple
from tqdm import tqdm
from google.colab import userdata

# Set your HuggingFace token for model access
# You can get one at: https://huggingface.co/settings/tokens
os.environ["HF_TOKEN"] = "hf_"  # Replace with your token

# Opik configuration (optional - for generation evaluation)
# Get your API key at: https://www.comet.com/
os.environ["OPIK_API_KEY"] = "4p"  # Replace with your Opik API key if available

# OpenAI API key (for proprietary models)
# Add your OpenAI API key to Colab secrets with the name OPENAI_API_KEY
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = userdata.get("test_key") # Changed to retrieve 'test_key'

print("Environment configured!")

Environment configured!


## 2. Data Loading

Loaded the HuggingFace documentation dataset.

In [ ]:
from datasets import load_dataset

# Load the HuggingFace documentation dataset
dataset = load_dataset("m-ric/huggingface_doc", split="train")

print(f"Dataset loaded with {len(dataset)} documents")
print(f"Columns: {dataset.column_names}")
print(f"\nSample document (first 500 chars):")
print(dataset[0]["text"][:500])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


huggingface_doc.csv:   0%|          | 0.00/22.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2647 [00:00<?, ? examples/s]

Dataset loaded with 2647 documents
Columns: ['text', 'source']

Sample document (first 500 chars):
 Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deploy [distilbert-base-uncased-finetuned-sst-2-english](https://huggingface.co/distilbert-base-uncased-finetuned-sst-2-english) for text classification. 

## 1. Enter the Hugging Face Repository ID and your desired endpoint name:

<img src="https://raw.githubusercontent.com/huggingface/hf-endpoints-docu


In [ ]:
# Extract text and source information
documents = []
for item in dataset:
    documents.append({
        "text": item["text"],
        "source": item["source"]
    })

print(f"Extracted {len(documents)} documents")

# For this assignment, we'll use a subset to keep things manageable
MAX_DOCS = 500
documents = documents[:MAX_DOCS]
print(f"Using {len(documents)} documents for this assignment")

Extracted 2647 documents
Using 500 documents for this assignment


## 3. Chunking

Splitted documents into smaller chunks for better retrieval.


Implemented the `chunk_document` function that:
1. Takes a text string, chunk_size, and chunk_overlap as parameters
2. Splits the text into overlapping chunks of the specified size
3. Returns a list of chunk strings


- Used a sliding window approach with step = chunk_size - chunk_overlap
- Handled edge cases: empty text, text shorter than chunk_size
- Make sure each chunk is non-empty before adding it

In [ ]:
def chunk_document(
    text: str,
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[str]:

    # Handle edge cases
    if not text:
        return []

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller than chunk_size"
        )

    if len(text) <= chunk_size:
        return [text]

    chunks = []

    # Sliding-window step
    step = chunk_size - chunk_overlap

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk)

        start += step

    return chunks


def chunk_all_documents(
    documents: List[Dict],
    chunk_size: int = 1000,
    chunk_overlap: int = 200
) -> List[Dict]:

    all_chunks = []
    chunk_id = 0

    for document in documents:

        text = document["text"]
        source = document["source"]

        chunks = chunk_document(
            text,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        for chunk in chunks:

            all_chunks.append({
                "chunk_id": chunk_id,
                "text": chunk,
                "source": source
            })

            chunk_id += 1

    return all_chunks

In [ ]:
# Test your chunking implementation
test_text = "A" * 2500  # 2500 characters
test_chunks = chunk_document(test_text, chunk_size=1000, chunk_overlap=200)

print(f"Test: 2500 char text with chunk_size=1000, overlap=200")
print(f"Expected chunks: ~4")
print(f"Your chunks: {len(test_chunks)}")

if len(test_chunks) >= 3 and len(test_chunks) <= 5:
    print("✅ Chunking test passed!")
else:
    print("❌ Check your chunking implementation")

Test: 2500 char text with chunk_size=1000, overlap=200
Expected chunks: ~4
Your chunks: 4
✅ Chunking test passed!


In [ ]:
# Create chunks from all documents
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

chunks = chunk_all_documents(documents, CHUNK_SIZE, CHUNK_OVERLAP)

print(f"\nCreated {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunks per document: {len(chunks) / len(documents):.2f}")

# Show sample chunk
if chunks:
    print(f"\nSample chunk:")
    print(f"  ID: {chunks[0]['chunk_id']}")
    print(f"  Source: {chunks[0]['source']}")
    print(f"  Text (first 200 chars): {chunks[0]['text'][:200]}...")


Created 5651 chunks from 500 documents
Average chunks per document: 11.30

Sample chunk:
  ID: 0
  Source: huggingface/hf-endpoints-documentation/blob/main/docs/source/guides/create_endpoint.mdx
  Text (first 200 chars):  Create an Endpoint

After your first login, you will be directed to the [Endpoint creation page](https://ui.endpoints.huggingface.co/new). As an example, this guide will go through the steps to deplo...


## 4. Embeddings

Generated vector embeddings for each chunk using BGE-small-en-v1.5.


Implemented the `generate_embeddings` function that:
1. Processes texts in batches for memory efficiency
2. Uses the SentenceTransformer model to generate embeddings
3. Returns embeddings as a list of lists (for Milvus compatibility)

- Used `model.encode()` with `normalize_embeddings=True` for cosine similarity
- Processed in batches to avoid memory issues
- Converted numpy arrays to lists using `.tolist()`

In [ ]:
from sentence_transformers import SentenceTransformer

# Load the embedding model
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" #Use any model of your choice from Sentence Transformers
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print(f"Loaded embedding model: {EMBEDDING_MODEL}")

# Test embedding
test_embedding = embedding_model.encode(["This is a test"], normalize_embeddings=True)
EMBEDDING_DIM = len(test_embedding[0])
print(f"Embedding dimension: {EMBEDDING_DIM}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

ImportError: cannot import name 'cached_download' from 'huggingface_hub' (/usr/local/lib/python3.13/dist-packages/huggingface_hub/__init__.py)

In [ ]:
from sentence_transformers import SentenceTransformer

# Re-testing embedding generation after reinstallation
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded embedding model: {EMBEDDING_MODEL}")
test_embedding = embedding_model.encode(["This is a test"], normalize_embeddings=True)
EMBEDDING_DIM = len(test_embedding[0])
print(f"Embedding dimension: {EMBEDDING_DIM}")

test_texts = ["Hello world", "This is a test", "RAG is cool"]
test_embeddings = generate_embeddings(test_texts, embedding_model)

print(f"\nGenerated {len(test_embeddings)} embeddings")
print(f"Embedding dimension: {len(test_embeddings[0]) if test_embeddings else 0}")

if len(test_embeddings) == 3 and len(test_embeddings[0]) == 384:
    print("✅ Embedding generation test passed!")
else:
    print("❌ Check your embedding implementation")

In [ ]:
def generate_embeddings(
    texts: List[str],
    model: SentenceTransformer,
    batch_size: int = 32
) -> List[List[float]]:
    """
    Generate embeddings for a list of texts.

    Args:
        texts: List of text strings to embed
        model: SentenceTransformer model
        batch_size: Number of texts to process at once

    Returns:
        List of embedding vectors (as lists of floats)
    """

    all_embeddings = []

    # Loop through texts in batches
    for i in tqdm(range(0, len(texts), batch_size)):

        # Get current batch
        batch = texts[i:i + batch_size]

        # Generate embeddings for this batch
        batch_embeddings = model.encode(
            batch,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        # Convert numpy array to Python list and add to result
        all_embeddings.extend(batch_embeddings.tolist())

    return all_embeddings

In [ ]:
# Test your embedding generation
test_texts = ["Hello world", "This is a test", "RAG is cool"]
test_embeddings = generate_embeddings(test_texts, embedding_model)

print(f"Generated {len(test_embeddings)} embeddings")
print(f"Embedding dimension: {len(test_embeddings[0]) if test_embeddings else 0}")

if len(test_embeddings) == 3 and len(test_embeddings[0]) == 384:
    print("✅ Embedding generation test passed!")
else:
    print("❌ Check your embedding implementation")

In [ ]:
# Generate embeddings for all chunks
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = generate_embeddings(chunk_texts, embedding_model)

print(f"\nGenerated {len(embeddings)} embeddings")
if embeddings:
    print(f"Embedding dimension: {len(embeddings[0])}")
    print(f"Sample embedding (first 10 values): {embeddings[0][:10]}")

## 5. Vector Store (Milvus)

Stored embeddings in Milvus for efficient similarity search.


1. Implemented `setup_milvus_collection` to create a new collection
2. Implementedinsert_data_to_milvus` to insert chunks and embeddings

->
- Used `client.has_collection()` to check if collection exists
- Used `client.drop_collection()` to remove existing collection
- Used `client.create_collection()` with dimension and metric_type parameters
- Used `client.insert()` to add data

In [ ]:
pip install pymilvus[milvus_lite]

In [ ]:
from pymilvus import MilvusClient

# Initialize Milvus client (uses Milvus Lite - stores data locally)
MILVUS_DB_PATH = "./hf_docs_milvus.db"
milvus_client = MilvusClient(uri=MILVUS_DB_PATH)

COLLECTION_NAME = "hf_documentation"

print(f"Milvus client initialized with database: {MILVUS_DB_PATH}")

In [ ]:
import os

# Remove Milvus database files for a clean restart
if os.path.exists(MILVUS_DB_PATH):
    os.remove(MILVUS_DB_PATH)
    print(f"Removed existing Milvus database file: {MILVUS_DB_PATH}")
if os.path.exists(MILVUS_DB_PATH + '.lock'):
    os.remove(MILVUS_DB_PATH + '.lock')
    print(f"Removed existing Milvus database lock file: {MILVUS_DB_PATH}.lock")

# Re-initialize Milvus client
milvus_client = MilvusClient(uri=MILVUS_DB_PATH)
print(f"Milvus client re-initialized with database: {MILVUS_DB_PATH}")


In [ ]:
# ============================================================
# TODO: IMPLEMENT MILVUS COLLECTION SETUP (10 points)
# ============================================================

def setup_milvus_collection(client: MilvusClient, collection_name: str, embedding_dim: int):
    """
    Create a Milvus collection for storing document embeddings.

    Args:
        client: MilvusClient instance
        collection_name: Name of the collection to create
        embedding_dim: Dimension of the embedding vectors
    """
    # TODO: Create a Milvus collection
    #
    # Step 1: Check if collection already exists using client.has_collection()
    # Step 2: If exists, drop it using client.drop_collection()
    # Step 3: Create new collection using client.create_collection() with:
    #   - collection_name: the name parameter
    #   - dimension: embedding_dim parameter
    #   - metric_type: "IP" (Inner Product for cosine similarity)
    #   - consistency_level: "Strong"

    # YOUR CODE HERE
    if client.has_collection(collection_name=collection_name):
        client.drop_collection(collection_name=collection_name)
        print(f"Dropped existing collection: {collection_name}")

    client.create_collection(
        collection_name=collection_name,
        dimension=embedding_dim,
        metric_type="IP", # Inner Product for cosine similarity
        consistency_level="Strong"
    )

    print(f"Created collection: {collection_name} with dimension {embedding_dim}")

In [ ]:
# Setup the collection
setup_milvus_collection(milvus_client, COLLECTION_NAME, EMBEDDING_DIM)

In [ ]:
# ============================================================
# TODO: IMPLEMENT DATA INSERTION (10 points)
# ============================================================

def insert_data_to_milvus(
    client: MilvusClient,
    collection_name: str,
    chunks: List[Dict],
    embeddings: List[List[float]],
    batch_size: int = 100
):
    """
    Insert document chunks and embeddings into Milvus.

    Args:
        client: MilvusClient instance
        collection_name: Name of the collection
        chunks: List of chunk dictionaries with text and metadata
        embeddings: List of embedding vectors
        batch_size: Number of records to insert at once

    Returns:
        Total number of inserted records
    """
    total_inserted = 0

    # TODO: Insert data into Milvus
    #
    # Step 1: Prepare data as a list of dictionaries, where each dict has:
    #   - "id": chunk["chunk_id"]
    #   - "vector": the corresponding embedding
    #   - "text": chunk["text"]
    #   - "source": chunk["source"]

    # Step 2: Insert in batches using client.insert()
    # Hint: Loop through data in batches and call:
    #   result = client.insert(collection_name=collection_name, data=batch)
    #   total_inserted += result["insert_count"]

    # YOUR CODE HERE
    data_to_insert = []
    for i in range(len(chunks)):
        data_to_insert.append({
            "id": chunks[i]["chunk_id"],
            "vector": embeddings[i],
            "text": chunks[i]["text"],
            "source": chunks[i]["source"]
        })

    for i in tqdm(range(0, len(data_to_insert), batch_size), desc="Inserting data to Milvus"):
        batch = data_to_insert[i:i + batch_size]
        result = client.insert(collection_name=collection_name, data=batch)
        total_inserted += result["insert_count"]

    return total_inserted

In [ ]:
# Insert data into Milvus
inserted_count = insert_data_to_milvus(milvus_client, COLLECTION_NAME, chunks, embeddings)

print(f"\nInserted {inserted_count} records into Milvus")

if inserted_count == len(chunks):
    print("✅ All chunks inserted successfully!")
else:
    print("❌ Not all chunks were inserted. Check your implementation.")

## 6. Retrieval

Implemented semantic search to retrieve relevant documents for a query.

->
Implement the `retrieve_documents` function that:
1. Generates an embedding for the query
2. Searches Milvus for similar vectors
3. Returns the top-K most relevant documents

->
- Use `embedding_model.encode()` to embed the query
- Use `client.search()` to find similar vectors
- Extract text and source from the search results

In [ ]:
# ============================================================
# TODO: IMPLEMENT RETRIEVAL (25 points)
# ============================================================

def retrieve_documents(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    top_k: int = 5
) -> List[Dict]:
    """
    Retrieve the most relevant documents for a query.

    Args:
        query: The search query
        client: MilvusClient instance
        collection_name: Name of the collection to search
        embedding_model: Model to generate query embedding
        top_k: Number of results to return

    Returns:
        List of dictionaries with 'text', 'source', 'chunk_id', and 'score' keys
    """
    # TODO: Implement semantic search
    #
    # Step 1: Generate embedding for the query
    # Hint: Use embedding_model.encode([query], normalize_embeddings=True)
    #       Then convert to list: .tolist()[0]

    # Step 2: Search in Milvus using client.search()
    # Required parameters:
    #   - collection_name: collection_name
    #   - data: [query_embedding] (list containing the embedding)
    #   - limit: top_k
    #   - search_params: {"metric_type": "IP", "params": {}}
    #   - output_fields: ["text", "source"]

    # Step 3: Format results as list of dicts
    # Each dict should have:
    #   - "text": result["entity"]["text"]
    #   - "source": result["entity"]["source"]
    #   - "chunk_id": result["id"]
    #   - "score": result["distance"]

    # YOUR CODE HERE
    retrieved_docs = []

    # Step 1: Generate embedding for the query
    query_embedding = embedding_model.encode([query], normalize_embeddings=True).tolist()[0]

    # Step 2: Search in Milvus
    search_results = client.search(
        collection_name=collection_name,
        data=[query_embedding],
        limit=top_k,
        search_params={"metric_type": "IP", "params": {}},
        output_fields=["text", "source"]
    )

    # Step 3: Format results
    for hit in search_results[0]: # MilvusClient.search returns a list of result sets
        retrieved_docs.append({
            "text": hit.entity["text"],   # Access entity as attribute, then key for text
            "source": hit.entity["source"], # Access entity as attribute, then key for source
            "chunk_id": hit.id,             # Access id as attribute
            "score": hit.distance           # Access distance as attribute
        })

    return retrieved_docs

In [ ]:
# Test retrieval
test_query = "How do I fine-tune a transformer model?"

retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved)} documents:")
for i, doc in enumerate(retrieved):
    print(f"\n--- Document {i+1} (Score: {doc.get('score', 'N/A')}) ---")
    print(f"Source: {doc.get('source', 'N/A')}")
    print(f"Text: {doc.get('text', 'N/A')[:300]}...")

if len(retrieved) == 3 and all('text' in d for d in retrieved):
    print("\n✅ Retrieval test passed!")
else:
    print("\n❌ Check your retrieval implementation")

*italicized text*## 7. Generation

Generated answers using Microsoft Phi-3-mini-4k-instruct/Qwen.

->
Implement the `generate_answer` function that:
1. Combines retrieved documents into a context string
2. Formats the prompt using the provided template
3. Generates an answer using the language model
4. Returns a structured result dictionary

->
- Join document texts with newlines to create context
- Use the PROMPT_TEMPLATE.format() to fill in context and question
- Call the generator pipeline with appropriate parameters

### https://huggingface.co/microsoft/Phi-3-mini-4k-instruct

### https://huggingface.co/microsoft/Phi-3.5-mini-instruct

### https://huggingface.co/Qwen/Qwen2-1.5B-Instruct

In [ ]:
%pip install -q openai

In [ ]:
# from google.colab import userdata
# from openai import OpenAI

# # Get API key securely from Colab Secrets
# OPENAI_API_KEY = userdata.get("test_key")

# # Create OpenAI client
# client = OpenAI(api_key=OPENAI_API_KEY)

# print("OpenAI client initialized successfully!")

###  USED A PROPRIETARY MODEL LIKE OPENAI, CLAUDE

In [ ]:
from google.colab import userdata
from openai import OpenAI

# Get OpenAI API key from Colab Secrets
OPENAI_API_KEY = userdata.get("test_key")

# Initialize OpenAI client
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# OpenAI proprietary model
LLM_MODEL = "gpt-5.6-luna"

print(f"Using OpenAI model: {LLM_MODEL}")

In [ ]:
def generate_response(prompt: str) -> str:

    response = openai_client.responses.create(
        model=LLM_MODEL,
        input=prompt
    )

    return response.output_text

In [ ]:
test_prompt = "What is Retrieval Augmented Generation?"

answer = generate_response(test_prompt)

print(answer)

### MODIFY THIS TO SUIT YOUR MODEL

In [ ]:
PROMPT_TEMPLATE = """You are a question-answering assistant for a specific knowledge base.

Your task is to answer the question using ONLY the information provided in <context>.

STRICT RULES:
1. Use only the information in <context>.
2. Do not use your own prior knowledge.
3. Do not make assumptions or fill in missing information.
4. If the answer cannot be found in <context>, respond exactly:
"I don't have enough information to answer this question."

<context>
{context}
</context>

<question>
{question}
</question>

Answer:"""

In [ ]:
# ============================================================
# TODO: IMPLEMENT GENERATION (25 points)
# ============================================================

def generate_answer(
    query: str,
    retrieved_docs: List[Dict],
    openai_client: OpenAI, # Changed from generator to openai_client
    llm_model: str,        # Added llm_model parameter
    max_new_tokens: int = 256
) -> Dict:
    """
    Generate an answer using retrieved documents as context.

    Args:
        query: The user's question
        retrieved_docs: List of retrieved document dictionaries
        openai_client: OpenAI client instance
        llm_model: Name of the OpenAI model to use
        max_new_tokens: Maximum tokens to generate

    Returns:
        Dictionary with 'answer', 'context', 'query', and 'retrieved_docs'
    """
    # TODO: Generate an answer using the RAG pattern
    #
    # Step 1: Combine retrieved documents into context
    # Hint: Join doc["text"] for each doc with "\n\n" separator

    # Step 2: Format the prompt using PROMPT_TEMPLATE
    # Hint: prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    # Step 3: Generate response using the OpenAI chat completion API
    # Call openai_client.chat.completions.create() with:
    #   - model: llm_model
    #   - messages: A list of dicts: [{'role': 'user', 'content': prompt}]
    #   - max_tokens: max_new_tokens
    #   - temperature: 0.7

    # Step 4: Extract the generated text
    # Hint: response.choices[0].message.content.strip()

    # Step 5: Return result dictionary

    # YOUR CODE HERE
    context = "\n\n".join([doc["text"] for doc in retrieved_docs])
    prompt = PROMPT_TEMPLATE.format(context=context, question=query)

    response = openai_client.chat.completions.create(
        model=llm_model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_completion_tokens=max_new_tokens,

    )

    answer = response.choices[0].message.content.strip()

    return {
        "query": query,
        "answer": answer,
        "context": context,
        "retrieved_docs": retrieved_docs
    }

In [ ]:
# Test generation
test_query = "How do I fine-tune a transformer model?"

# Retrieve relevant documents
retrieved = retrieve_documents(
    query=test_query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=3
)

# Generate answer
result = generate_answer(
    query=test_query,
    retrieved_docs=retrieved,
    openai_client=openai_client, # Changed from generator
    llm_model=LLM_MODEL          # Added LLM_MODEL
)

print(f"Question: {result['query']}")
print(f"\nAnswer: {result['answer']}")

if result['answer'] and len(result['answer']) > 10:
    print("\n✅ Generation test passed!")
else:
    print("\n❌ Check your generation implementation")

In [ ]:
# Complete RAG pipeline function (DO NOT MODIFY)

def rag_query(
    query: str,
    client: MilvusClient,
    collection_name: str,
    embedding_model: SentenceTransformer,
    openai_client: OpenAI, # Changed from generator
    llm_model: str,        # Added llm_model
    top_k: int = 5,
    max_new_tokens: int = 256
) -> Dict:
    """
    Complete RAG pipeline: retrieve then generate.
    """
    # Retrieve
    retrieved_docs = retrieve_documents(
        query=query,
        client=client,
        collection_name=collection_name,
        embedding_model=embedding_model,
        top_k=top_k
    )

    # Generate
    result = generate_answer(
        query=query,
        retrieved_docs=retrieved_docs,
        openai_client=openai_client, # Changed from generator
        llm_model=llm_model,        # Added llm_model
        max_new_tokens=max_new_tokens
    )

    return result

In [ ]:
# Test complete pipeline with multiple queries
test_queries = [
    "What is the Trainer class in transformers?",
    "How do I load a dataset from HuggingFace?",
    "What is Gradio used for?",
    "what is the capital of France?"
]

for query in test_queries:
    print(f"\n{'='*60}")
    result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        openai_client=openai_client, # Changed from generator
        llm_model=LLM_MODEL          # Added LLM_MODEL
    )
    print(f"Q: {result['query']}")
    print(f"A: {result['answer']}")

## 8. Evaluation

Evaluated the performance of the RAG pipeline using Opik metrics and traditional retrieval metrics.

In [ ]:
# ============================================================
# TASK 5.1: BUILD LABELED TEST SET
# ============================================================

test_queries = [
    {
        "query": "What is the Trainer class in Transformers?",
        "keywords": ["Trainer", "training"]
    },
    {
        "query": "How do I load a dataset from Hugging Face?",
        "keywords": ["load_dataset", "dataset"]
    },
    {
        "query": "What is Gradio used for?",
        "keywords": ["Gradio", "interface"]
    },
    {
        "query": "How do I create an endpoint?",
        "keywords": ["create", "endpoint"]
    },
    {
        "query": "How can I fine-tune a transformer model?",
        "keywords": ["fine-tune", "training"]
    }
]

In [ ]:
def find_candidate_chunks(chunks, keywords, top_n=5):
    """
    Find chunks containing the specified keywords.
    Used to manually identify ground-truth relevant chunks.
    """

    candidates = []

    for chunk in chunks:
        text_lower = chunk["text"].lower()

        # Count how many keywords appear in the chunk
        score = sum(
            1 for keyword in keywords
            if keyword.lower() in text_lower
        )

        if score > 0:
            candidates.append({
                "chunk_id": chunk["chunk_id"],
                "source": chunk["source"],
                "keyword_matches": score,
                "text": chunk["text"]
            })

    # Highest keyword match first
    candidates.sort(
        key=lambda x: x["keyword_matches"],
        reverse=True
    )

    return candidates[:top_n]

In [ ]:
for item in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", item["query"])
    print("=" * 80)

    candidates = find_candidate_chunks(
        chunks,
        item["keywords"],
        top_n=3
    )

    for candidate in candidates:
        print("\nChunk ID:", candidate["chunk_id"])
        print("Source:", candidate["source"])
        print("Keyword matches:", candidate["keyword_matches"])
        print("Text:", candidate["text"][:500])

In [ ]:
def create_labeled_test_case(query, relevant_chunk_ids):
    return {
        "query": query,
        "relevant_chunk_ids": relevant_chunk_ids
    }


labeled_test_set = []

labeled_test_set.append(
    create_labeled_test_case(
        "What is the Trainer class in Transformers?",
        [3373, 209]   # Replace with your actual relevant IDs
    )
)

labeled_test_set.append(
    create_labeled_test_case(
        "How do I load a dataset from Hugging Face?",
        [362, 4172]   # Replace with your actual relevant IDs
    )
)

labeled_test_set.append(
    create_labeled_test_case(
        "What is Gradio used for?",
        [3503,1838]        # Replace with your actual relevant ID
    )
)

labeled_test_set.append(
    create_labeled_test_case(
        "How do I create an endpoint?",
        [0, 2] # Replace with your actual IDs
    )
)

labeled_test_set.append(
    create_labeled_test_case(
        "How can I fine-tune a transformer model?",
        [83, 135]   # Replace with your actual IDs
    )
)
labeled_test_set.append({
    "query": "What is the capital of France?",
    "relevant_chunk_ids": []
})

print("Number of test queries:", len(labeled_test_set))

for item in labeled_test_set:
    print("\nQuery:", item["query"])
    print("Relevant chunks:", item["relevant_chunk_ids"])

In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    """
    Precision@K = relevant retrieved chunks / K
    """
    retrieved_at_k = retrieved_ids[:k]

    if k == 0:
        return 0.0

    relevant_retrieved = len(
        set(retrieved_at_k) & set(relevant_ids)
    )

    return relevant_retrieved / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    """
    Recall@K = relevant retrieved chunks / total relevant chunks
    """
    retrieved_at_k = retrieved_ids[:k]

    if len(relevant_ids) == 0:
        return 0.0

    relevant_retrieved = len(
        set(retrieved_at_k) & set(relevant_ids)
    )

    return relevant_retrieved / len(relevant_ids)

In [ ]:
K_VALUES = [1, 3, 5]

evaluation_results = []

for test_case in labeled_test_set:

    query = test_case["query"]
    relevant_ids = test_case["relevant_chunk_ids"]

    # Retrieve documents using your existing RAG retriever
    retrieved_docs = retrieve_documents(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=max(K_VALUES)
    )

    # Extract chunk IDs
    retrieved_ids = [
        doc["chunk_id"]
        for doc in retrieved_docs
    ]

    # Generate answer
    generation_result = generate_answer(
        query=query,
        retrieved_docs=retrieved_docs,
        openai_client=openai_client,
        llm_model=LLM_MODEL
    )
    generated_answer_text = generation_result['answer']

    print(f"\nQuery: {query}")
    print(f"Relevant Chunk IDs (Ground Truth): {relevant_ids}")
    print(f"Retrieved Chunk IDs: {retrieved_ids}")
    print(f"Generated Answer: {generated_answer_text[:100]}...") # Print a snippet of the answer

    result = {
        "query": query,
        "relevant_chunk_ids": relevant_ids,
        "retrieved_chunk_ids": retrieved_ids,
        "generated_answer": generated_answer_text # Store the full generated answer
    }

    # Calculate metrics
    for k in K_VALUES:
        result[f"precision@{k}"] = precision_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )

        result[f"recall@{k}"] = recall_at_k(
            retrieved_ids,
            relevant_ids,
            k
        )

    evaluation_results.append(result)


In [ ]:
for result in evaluation_results:

    print("\n" + "=" * 80)
    print("Query:", result["query"])
    print("Answer:", result["generated_answer"])

    print(
        f"Precision@1: {result['precision@1']:.2f} | "
        f"Recall@1: {result['recall@1']:.2f}"
    )

    print(
        f"Precision@3: {result['precision@3']:.2f} | "
        f"Recall@3: {result['recall@3']:.2f}"
    )

    print(
        f"Precision@5: {result['precision@5']:.2f} | "
        f"Recall@5: {result['recall@5']:.2f}"
    )


In [ ]:
query = "What is Gradio used for?"

retrieved_docs = retrieve_documents(
    query=query,
    client=milvus_client,
    collection_name=COLLECTION_NAME,
    embedding_model=embedding_model,
    top_k=5
)

for i, doc in enumerate(retrieved_docs, 1):
    print("=" * 80)
    print("Rank:", i)
    print("Chunk ID:", doc["chunk_id"])
    print("Score:", doc["score"])
    print("Source:", doc["source"])
    print("Text:")
    print(doc["text"][:700])

In [ ]:
%pip install -q opik

In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("test_key")

print("OpenAI key configured:", bool(os.environ["OPENAI_API_KEY"]))

In [ ]:
from opik.evaluation.metrics import AnswerRelevance, Hallucination

answer_relevance_metric = AnswerRelevance()
hallucination_metric = Hallucination()

In [ ]:
evaluation_results_53 = []

for test_case in labeled_test_set:

    query = test_case["query"]

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    # Run your existing RAG pipeline
    result = rag_query(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        openai_client=openai_client,
        llm_model=LLM_MODEL
    )

    answer = result["answer"]

    # Retrieve context for evaluation
    retrieved_docs = retrieve_documents(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=5
    )

    context = [doc["text"] for doc in retrieved_docs]

    # Answer relevance
    relevance_score = answer_relevance_metric.score(
        input=query,
        output=answer,
        context=context
    )

    # Hallucination
    hallucination_score = hallucination_metric.score(
        input=query,
        output=answer,
        context=context
    )

    evaluation_results_53.append({
        "query": query,
        "answer": answer,
        "answer_relevance": relevance_score.value,
        "relevance_reason": relevance_score.reason,
        "hallucination": hallucination_score.value,
        "hallucination_reason": hallucination_score.reason
    })

    print("Answer:", answer)
    print("Answer Relevance:", relevance_score.value)
    print("Hallucination:", hallucination_score.value)

In [ ]:
# Task 5.4: Evaluation thresholds

RELEVANCE_THRESHOLD = 0.80
HALLUCINATION_THRESHOLD = 0.20

print("Evaluation thresholds:")
print(f"Answer Relevance >= {RELEVANCE_THRESHOLD} → Good")
print(f"Hallucination <= {HALLUCINATION_THRESHOLD} → Good")

In [ ]:
# Analyze answer quality

print("=" * 80)
print("TASK 5.4 - ANSWER QUALITY ANALYSIS")
print("=" * 80)

for result in evaluation_results_53:

    relevance = result["answer_relevance"]
    hallucination = result["hallucination"]

    relevance_status = (
        "PASS" if relevance >= RELEVANCE_THRESHOLD else "FAIL"
    )

    hallucination_status = (
        "PASS" if hallucination <= HALLUCINATION_THRESHOLD else "FAIL"
    )

    print(f"\nQuery: {result['query']}")
    print(f"Answer Relevance: {relevance:.2f} → {relevance_status}")
    print(f"Hallucination: {hallucination:.2f} → {hallucination_status}")

In [ ]:
# Analyze retrieval results
print("=" * 80)
print("TASK 5.4 - RETRIEVAL ANALYSIS")
print("=" * 80)

for result in evaluation_results:

    print(f"\nQuery: {result['query']}")

    for k in K_VALUES:
        precision = result[f"precision@{k}"]
        recall = result[f"recall@{k}"]

        print(
            f"Precision@{k}: {precision:.2f} | "
            f"Recall@{k}: {recall:.2f}"
        )

In [ ]:
RETRIEVAL_THRESHOLD = 0.50

In [ ]:
#Automatically identify low-scoring retrieval cases
low_retrieval_cases = []

for result in evaluation_results:

    # Use Recall@5 as the main retrieval coverage indicator
    recall_5 = result["recall@5"]

    if recall_5 < RETRIEVAL_THRESHOLD:
        low_retrieval_cases.append(result)

print("Low-scoring retrieval cases:")
print("=" * 80)

for result in low_retrieval_cases:
    print(
        f"{result['query']} "
        f"→ Recall@5 = {result['recall@5']:.2f}"
    )

In [ ]:
#Inspect the actual retrieved chunks
for result in low_retrieval_cases:

    query = result["query"]

    print("\n" + "=" * 100)
    print("QUERY:", query)
    print("=" * 100)

    retrieved_docs = retrieve_documents(
        query=query,
        client=milvus_client,
        collection_name=COLLECTION_NAME,
        embedding_model=embedding_model,
        top_k=5
    )

    for rank, doc in enumerate(retrieved_docs, 1):

        print("\n" + "-" * 80)
        print("Rank:", rank)
        print("Chunk ID:", doc["chunk_id"])
        print("Score:", round(doc["score"], 4))
        print("Source:", doc["source"])
        print("Text:")
        print(doc["text"][:700])

In [ ]:
#Create a final diagnosis table
# Create a concise diagnosis summary

print("=" * 100)
print("FINAL RAG DIAGNOSIS")
print("=" * 100)

for result in evaluation_results:

    query = result["query"]

    print(f"\nQuery: {query}")

    print(
        f"  Retrieval → "
        f"P@5={result['precision@5']:.2f}, "
        f"R@5={result['recall@5']:.2f}"
    )

    # Find corresponding answer evaluation
    answer_result = next(
        r for r in evaluation_results_53
        if r["query"] == query
    )

    print(
        f"  Generation → "
        f"Relevance={answer_result['answer_relevance']:.2f}, "
        f"Hallucination={answer_result['hallucination']:.2f}"
    )

## 9. Resources

Here are some useful resources and model links:

*   **Milvus**: https://github.com/milvus-io/milvus
*   **Microsoft Phi-3-mini-4k-instruct**: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct
*   **Microsoft Phi-3.5-mini-instruct**: https://huggingface.co/microsoft/Phi-3.5-mini-instruct
*   **Qwen/Qwen2-1.5B-Instruct**: https://huggingface.co/Qwen/Qwen2-1.5B-Instruct